In [44]:
import json
from typing import List, Tuple, Dict
from collections import defaultdict

In [46]:
CONDITIONS = {
    "condition_1": ["condition 1"],
    "condition_2": ["condition 2"],
    "condition_3": ["condition 3"],
}


In [48]:
# Convert HH:MM:SS timestamp to seconds
def time_to_seconds(timestamp: str) -> int:
    """
    Convert timestamp in HH:MM:SS format to total seconds.
    """
    hours, minutes, seconds = map(int, timestamp.split(":"))
    return hours * 3600 + minutes * 60 + seconds

In [50]:
# Load segments with condition classification
# ==========================================================
def load_segments_with_condition(path: str) -> List[Tuple[int, int, str]]:
    """
    Load segments from JSON file.
    Assign condition label based on justification text.
    Returns: (start_sec, end_sec, condition)
    """
    with open(path, "r") as f:
        data = json.load(f)

    segments = []

    for seg in data["segments"]:
        start = time_to_seconds(seg["time_in"])
        end = time_to_seconds(seg["time_out"])
        justification = seg.get("justification", "").lower()

        condition = None
        for cond, keywords in CONDITIONS.items():
            if any(keyword in justification for keyword in keywords):
                condition = cond
                break

        if end > start and condition:
            segments.append((start, end, condition))

    return segments

In [52]:
# Group segments by condition
def group_by_condition(
    segments: List[Tuple[int, int, str]]
) -> Dict[str, List[Tuple[int, int]]]:
    """
    Group segments by condition.
    Returns dict: {condition: [(start, end), ...]}
    """
    grouped = defaultdict(list)

    for start, end, condition in segments:
        grouped[condition].append((start, end))

    return grouped

In [54]:
# Merge overlapping intervals
def merge_intervals(segments: List[Tuple[int, int]]) -> List[Tuple[int, int]]:
    """
    Merge overlapping time intervals.
    """
    if not segments:
        return []

    segments = sorted(segments)
    merged = [segments[0]]

    for current_start, current_end in segments[1:]:
        last_start, last_end = merged[-1]

        if current_start <= last_end:
            merged[-1] = (last_start, max(last_end, current_end))
        else:
            merged.append((current_start, current_end))

    return merged

In [56]:
# Compute total temporal coverage
def total_coverage(segments: List[Tuple[int, int]]) -> int:
    """
    Compute total covered duration after merging overlaps.
    """
    merged = merge_intervals(segments)
    return sum(end - start for start, end in merged)

In [58]:
##average segment length+coverage density(total coverage/total timeline length)
gpt4_path = "gpt4.json"      
gpt52_path = "gpt5.2.json"    

def merge_intervals_old(segments):
    intervals = sorted([(start, end) for start, end, _ in segments])
    
    merged = []
    for start, end in intervals:
        if not merged or start > merged[-1][1]:
            merged.append([start, end])
        else:
            merged[-1][1] = max(merged[-1][1], end)
    
    return merged

def compute_metrics(path):
    segments = load_segments_with_condition(path)

    segment_count = len(segments)

    # Remove condition label
    intervals = [(start, end) for start, end, _ in segments]

    merged = merge_intervals(intervals)
    total_cov = sum(end - start for start, end in merged)

    avg_segment_length = (
        total_cov / segment_count
        if segment_count > 0 else 0
    )

    if intervals:
        timeline_start = min(start for start, _ in intervals)
        timeline_end = max(end for _, end in intervals)
        timeline_length = timeline_end - timeline_start

        coverage_density = (
            total_cov / timeline_length
            if timeline_length > 0 else 0
        )
    else:
        coverage_density = 0

    return segment_count, total_cov, avg_segment_length, coverage_density


gpt4_metrics = compute_metrics(gpt4_path)
gpt52_metrics = compute_metrics(gpt52_path)



print("===== GPT-4 =====")
print("Segment Count:", gpt4_metrics[0])
print("Total Coverage:", gpt4_metrics[1])
print("Average Segment Length:", gpt4_metrics[2])
print("Coverage Density:", gpt4_metrics[3])

print("\n===== GPT-5.2 =====")
print("Segment Count:", gpt52_metrics[0])
print("Total Coverage:", gpt52_metrics[1])
print("Average Segment Length:", gpt52_metrics[2])
print("Coverage Density:", gpt52_metrics[3])


===== GPT-4 =====
Segment Count: 43
Total Coverage: 316
Average Segment Length: 7.348837209302325
Coverage Density: 0.5205930807248764

===== GPT-5.2 =====
Segment Count: 189
Total Coverage: 570
Average Segment Length: 3.015873015873016
Coverage Density: 0.9283387622149837


In [60]:
# Compute intersection time
def intersection_time(
    segments1: List[Tuple[int, int]],
    segments2: List[Tuple[int, int]]
) -> int:
    """
    Compute total overlapping duration between two segment sets.
    """
    seg1 = merge_intervals(segments1)
    seg2 = merge_intervals(segments2)

    i, j = 0, 0
    intersection = 0

    while i < len(seg1) and j < len(seg2):
        start1, end1 = seg1[i]
        start2, end2 = seg2[j]

        start = max(start1, start2)
        end = min(end1, end2)

        if start < end:
            intersection += end - start

        if end1 < end2:
            i += 1
        else:
            j += 1

    return intersection

In [62]:
# Compute IoU (Intersection over Union)
def compute_iou(
    segments1: List[Tuple[int, int]],
    segments2: List[Tuple[int, int]]
) -> float:
    """
    Compute temporal IoU between two segment sets.
    """
    intersection = intersection_time(segments1, segments2)
    union = (
        total_coverage(segments1)
        + total_coverage(segments2)
        - intersection
    )

    if union == 0:
        return 0.0

    return intersection / union

In [64]:
# Compute average boundary deviation
def boundary_deviation(
    segments1: List[Tuple[int, int]],
    segments2: List[Tuple[int, int]]
) -> float:
    """
    Compute average absolute boundary deviation
    between overlapping segments.
    """
    deviations = []

    for s1, e1 in segments1:
        for s2, e2 in segments2:
            overlap = min(e1, e2) - max(s1, s2)

            if overlap > 0:
                deviations.append(abs(s1 - s2))
                deviations.append(abs(e1 - e2))

    if not deviations:
        return 0.0

    return sum(deviations) / len(deviations)

In [66]:
# RUN EVALUATION
# ==========================================================

gpt4_segments = load_segments_with_condition("gpt4.json")
gpt5_segments = load_segments_with_condition("gpt5.2.json")

gpt4_grouped = group_by_condition(gpt4_segments)
gpt5_grouped = group_by_condition(gpt5_segments)

print("\n========== PER-CONDITION EVALUATION ==========")

for condition in gpt4_grouped:
    if condition in gpt5_grouped:

        seg1 = gpt4_grouped[condition]
        seg2 = gpt5_grouped[condition]

        print(f"\nCondition: {condition}")
        print("Segment Count GPT-4:", len(seg1))
        print("Segment Count GPT-5.2:", len(seg2))
        print("Total Coverage GPT-4:", total_coverage(seg1))
        print("Total Coverage GPT-5.2:", total_coverage(seg2))
        print("Intersection:", intersection_time(seg1, seg2))
        print("IoU:", compute_iou(seg1, seg2))
        print("Boundary Deviation:", boundary_deviation(seg1, seg2))


========== PER-CONDITION EVALUATION ==========

Condition: condition_1
Segment Count GPT-4: 14
Segment Count GPT-5.2: 26
Total Coverage GPT-4: 188
Total Coverage GPT-5.2: 379
Intersection: 188
IoU: 0.49604221635883905
Boundary Deviation: 0.0

Condition: condition_2
Segment Count GPT-4: 7
Segment Count GPT-5.2: 109
Total Coverage GPT-4: 68
Total Coverage GPT-5.2: 548
Intersection: 68
IoU: 0.12408759124087591
Boundary Deviation: 55.083657587548636

Condition: condition_3
Segment Count GPT-4: 22
Segment Count GPT-5.2: 54
Total Coverage GPT-4: 292
Total Coverage GPT-5.2: 526
Intersection: 292
IoU: 0.5551330798479087
Boundary Deviation: 38.85070140280561
